# Machine Learning Engineer Capstone: Moisture Content
**Objective:** Construct an end-to-end machine learning workflow to predict humidity and temperature's effects on moisture content.

## Required Libraries

In [2]:
# ... for data science
import numpy as np
import pandas as pd

In [3]:
# ... for machine learning
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Lasso, Ridge
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.decomposition import PCA
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

## EDA for Linear Regression

In [4]:
df = pd.read_csv('0000 Devens Logger Data.csv', header=1) # because the first row contains group labels while the second contains the actual column names

C:\Windows\Temp\ipykernel_28360\3352695845.py:1: DtypeWarning: Columns (0: Unnamed: 200, 1: N1-MCEM-MC.1, 2: N2-MCEM-MC.1, 3: N3-MCEM-MC.1, 4: Unnamed: 204, 5: Unnamed: 205, 6: S1-MCEM-MC.1, 7: S2-MCEM-MC.1, 8: S3-MCEM-MC.1, 9: Unnamed: 209, 10: Unnamed: 210, 11: Unnamed: 214, 12: Unnamed: 218, 13: Unnamed: 219, 14: N1 T 30 Day Average, 15: N2 T 30 Day Average, 16: N3 T 30 Day Average, 17: Unnamed: 229, 18: Sheath T (30 Day Avg), 19: Unnamed: 234, 20: S1 T 30 Day Average) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('0000 Devens Logger Data.csv', header=1) # because the first row contains group labels while the second contains the actual column names


In [5]:
# Preview the dataset
df.head()

,DateAndTime,T out,RH out,RH out-C,RH out-DP,N1-TCEM,N1-TCNM,N1-TCIM,N1-TENU,N1-TENM,...,S2 T 30 Day Average,S2 Pass-Fail,S3 RH 30 Day Average,S3 T 30 Day Average,S3 Pass-Fail,Unnamed: 244,S1.1,S2.1,S3.1,Sheath T (30 Day Avg).1
0,12/06/2011 20:00,NaN,11.4,11.4,-105.3,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,12/07/2011 15:00,NaN,21.3,21.3,-110.5,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,12/07/2011 16:00,NaN,54.5,54.5,-55.6,10.3,13.7,16.8,8.4,8.4,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,12/07/2011 17:00,NaN,16.0,16.0,-57.4,10.0,13.4,16.7,8.1,8.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,12/07/2011 18:00,NaN,53.1,53.1,-39.8,9.8,13.3,16.8,7.9,7.9,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
# Split dataset into two -- north and south
df_north = df.filter(regex=r'^N[123]-').copy()
df_north[['T out', 'RH out', 'RH out-DP']] = df[['T out', 'RH out', 'RH out-DP']]
df_south = df.filter(regex=r'^S[123]-').copy()
df_south[['T out', 'RH out', 'RH out-DP']] = df[['T out', 'RH out', 'RH out-DP']]

In [7]:
# Data types
print('North side data types:')
print(df_north.dtypes)
print('\nSouth side data types:')
print(df_south.dtypes)

North side data types:
N1-TCEM         float64
N1-TCNM         float64
N1-TCIM         float64
N1-TENU         float64
N1-TENM         float64
                 ...   
N2-MCEM-MC.1     object
N3-MCEM-MC.1     object
T out           float64
RH out          float64
RH out-DP       float64
Length: 91, dtype: object

South side data types:
S1-TCEM         float64
S1-TCNM         float64
S1-TCIM         float64
S1-TENU         float64
S1-TENM         float64
                 ...   
S2-MCEM-MC.1     object
S3-MCEM-MC.1     object
T out           float64
RH out          float64
RH out-DP       float64
Length: 92, dtype: object


**Verdict:** The majority of variables in both subsets are floats, so it makes sense to apply a regression algorithm.

In [8]:
# Summary statistics
print('Summary statistics from north side:')
print(df_north.describe())
print('\nSummary statistics from north side:')
print(df_south.describe())

Summary statistics from north side:
            N1-TCEM       N1-TCNM       N1-TCIM       N1-TENU       N1-TENM  \
count  22861.000000  22861.000000  22861.000000  22861.000000  22861.000000   
mean      12.097804     16.092734     20.583890     10.689056     10.185434   
std        9.863197      6.373062      3.028832     11.572328     11.830849   
min      -14.100000      0.700000     10.400000    -21.300000    -23.100000   
25%        4.200000     10.900000     18.300000      1.500000      1.100000   
50%       11.400000     15.300000     20.000000     10.000000      9.500000   
75%       20.000000     21.400000     22.800000     19.700000     19.300000   
max       36.700000     32.500000     31.300000     39.300000     42.100000   

            N1-TENL       N1-TDEM       N1-TDIM       N1-TAEM       N2-TCEM  \
count  22861.000000  22861.000000  22861.000000  22861.000000  22861.000000   
mean       9.999077     10.629203     12.148078     21.579612     11.917318   
std       11.68

In [9]:
# Missing values
print('Missing values in north side:')
print(df_north.info())
print('\nMissing values in south side:')
print(df_south.info())

Missing values in north side:
<class 'pandas.DataFrame'>
RangeIndex: 22879 entries, 0 to 22878
Data columns (total 91 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   N1-TCEM       22861 non-null  float64
 1   N1-TCNM       22861 non-null  float64
 2   N1-TCIM       22861 non-null  float64
 3   N1-TENU       22861 non-null  float64
 4   N1-TENM       22861 non-null  float64
 5   N1-TENL       22861 non-null  float64
 6   N1-TDEM       22861 non-null  float64
 7   N1-TDIM       22861 non-null  float64
 8   N1-TAEM       22861 non-null  float64
 9   N2-TCEM       22861 non-null  float64
 10  N2-TCNM       22861 non-null  float64
 11  N2-TCIM       22861 non-null  float64
 12  N2-TENU       22861 non-null  float64
 13  N2-TENM       22861 non-null  float64
 14  N2-TENL       22861 non-null  float64
 15  N2-TDEM       22861 non-null  float64
 16  N2-TDIM       22861 non-null  float64
 17  N2-TAEM       22861 non-null  float64
 18  N3-TCEM

**Verdict:** Imputation is necessary for variables such as `N2-MENL`.

## Feature Engineering

In [10]:
# 1. Split north and south datasets into features and targets
moisture_regex = r'^[NS][123]-M.*-MC$'
temp_regex = r'^[NS][123]-[TR]'
y_north = df_north.filter(regex=moisture_regex).copy()
X_north = df_north.filter(regex=temp_regex).copy()
y_south = df_south.filter(regex=moisture_regex).copy()
X_south = df_south.filter(regex=temp_regex).copy()

In [11]:
print(X_north)
print(y_north)
print(X_south)
print(y_south)

       N1-TCEM  N1-TCNM  N1-TCIM  N1-TENU  N1-TENM  N1-TENL  N1-TDEM  N1-TDIM  \
0          NaN      NaN      NaN      NaN      NaN      NaN      NaN      NaN   
1          NaN      NaN      NaN      NaN      NaN      NaN      NaN      NaN   
2         10.3     13.7     16.8      8.4      8.4      8.2      9.2     10.9   
3         10.0     13.4     16.7      8.1      8.0      7.9      8.8     10.6   
4          9.8     13.3     16.8      7.9      7.9      7.8      8.7     10.4   
...        ...      ...      ...      ...      ...      ...      ...      ...   
22874      NaN      NaN      NaN      NaN      NaN      NaN      NaN      NaN   
22875      NaN      NaN      NaN      NaN      NaN      NaN      NaN      NaN   
22876      NaN      NaN      NaN      NaN      NaN      NaN      NaN      NaN   
22877      NaN      NaN      NaN      NaN      NaN      NaN      NaN      NaN   
22878      NaN      NaN      NaN      NaN      NaN      NaN      NaN      NaN   

       N1-TAEM  N2-TCEM  ..

In [12]:
# 2. Define features (including outdoor conditions)
weather = ['T out', 'RH out', 'RH out-DP']
north_lag = y_north.shift(1) # the previous observation
north_previous_mean = north_lag.rolling(24).mean() # the average of the past 24 observations
X_north_lag, X_north_previous_mean = X_north.join(north_lag), X_north.join(north_previous_mean)
X_north_updated = pd.concat([X_north, north_lag.add_suffix('_lag_1'), north_previous_mean.add_suffix('_mean_previous_24')], axis=1)
X_north_updated[weather] = df_north[weather]
south_lag = y_south.shift(1)
south_previous_mean = south_lag.rolling(24).mean()
X_south_lag, X_south_previous_mean = X_south.join(south_lag), X_south.join(south_previous_mean)
X_south_updated = pd.concat([X_south, south_lag.add_suffix('_lag_1'), south_previous_mean.add_suffix('_mean_previous_24')], axis=1)
X_south_updated[weather] = df_south[weather]

In [13]:
print(y_north.isna().sum())
print(y_south.isna().sum())

N1-MENU-MC    339
N1-MENM-MC    364
N1-MENL-MC    211
N1-MDEM-MC     18
N1-MDIM-MC     18
N1-MCEM-MC     18
N2-MENU-MC    188
N2-MENM-MC     39
N2-MENL-MC     95
N2-MDEM-MC     20
N2-MDIM-MC     30
N2-MCEM-MC     19
N3-MENU-MC    545
N3-MENM-MC    827
N3-MENL-MC    694
N3-MDEM-MC     25
N3-MDIM-MC     36
N3-MCEM-MC     22
dtype: int64
S1-MENU-MC    3657
S1-MENM-MC    5024
S1-MENL-MC    7623
S1-MDEM-MC     341
S1-MDIM-MC      82
S1-MCEM-MC     924
S2-MENU-MC    3101
S2-MENM-MC    2626
S2-MENL-MC    9379
S2-MDEM-MC      58
S2-MDIM-MC      41
S2-MCEM-MC     346
S3-MENU-MC    2464
S3-MENM-MC    7720
S3-MENL-MC    5344
S3-MDEM-MC    1465
S3-MDIM-MC     567
S3-MCEM-MC    3681
dtype: int64


In [14]:
# 3. Remove missing values from the targets (and features, if necessary)
north_valid, south_valid = y_north.notna().all(axis=1), y_south.notna().all(axis=1)
X_north_updated = X_north_updated.loc[north_valid].copy()
y_north = y_north.loc[north_valid].copy()
X_south_updated = X_south_updated.loc[south_valid].copy()
y_south = y_south.loc[south_valid].copy()

In [15]:
# 4. Produce training/testing sets for both datasets
X_train_north, X_test_north, y_train_north, y_test_north = train_test_split(X_north_updated,
                                                                            y_north,
                                                                            random_state=42,
                                                                            test_size=0.20,
                                                                            shuffle=False)
X_train_south, X_test_south, y_train_south, y_test_south = train_test_split(X_south_updated,
                                                                            y_south,
                                                                            random_state=42,
                                                                            test_size=0.20,
                                                                            shuffle=False)

In [16]:
# 5a. Create a preprocessing pipeline that will ...
#    - Fill in missing values
#    - Scale columns
#    - Reduce features
preprocessor = Pipeline([('imputer', SimpleImputer(strategy='mean')),
                         ('scale', StandardScaler()),
                         ('pca', PCA(n_components=0.95))])

In [17]:
# 5b. Apply the preprocessor to the training and testing sets of both datasets
X_train_north_processed = preprocessor.fit_transform(X_train_north)
X_test_north_processed = preprocessor.transform(X_test_north)
X_train_south_processed = preprocessor.fit_transform(X_train_south)
X_test_south_processed = preprocessor.transform(X_test_south)

C:\Users\Michael\anaconda3\Lib\site-packages\sklearn\impute\_base.py:647: UserWarning: Skipping features without any observed values: ['S1-TAEM']. At least one non-missing value is needed for imputation with strategy='mean'.
  warnings.warn(
C:\Users\Michael\anaconda3\Lib\site-packages\sklearn\impute\_base.py:647: UserWarning: Skipping features without any observed values: ['S1-TAEM']. At least one non-missing value is needed for imputation with strategy='mean'.
  warnings.warn(


## Implement Machine Learning Workflow

In [18]:
mlr_raw = LinearRegression()

In [19]:
# 1. The north side
mlr_raw.fit(X_train_north_processed, y_train_north)
y_north_pred = mlr_raw.predict(X_test_north_processed)
MAE_north = mean_absolute_error(y_test_north, y_north_pred) # the lower the better
MSE_north = mean_squared_error(y_test_north, y_north_pred) # ditto
r2_north = r2_score(y_test_north, y_north_pred)
print(f'MAE on the north side: {MAE_north:.3f}')
print(f'MSE on the north side: {MSE_north:.3f}')
print(f'R2 on the north side: {r2_north:.3f}')

MAE on the north side: 0.624
MSE on the north side: 0.811
R2 on the north side: 0.776


In [20]:
# 2. The south side
mlr_raw.fit(X_train_south_processed, y_train_south)
y_south_pred = mlr_raw.predict(X_test_south_processed)
MAE_south = mean_absolute_error(y_test_south, y_south_pred) # the lower the better
MSE_south = mean_squared_error(y_test_south, y_south_pred) # ditto
r2_south = r2_score(y_test_south, y_south_pred)
print(f'MAE on the south side: {MAE_south:.3f}')
print(f'MSE on the south side: {MSE_south:.3f}')
print(f'R2 on the south side: {r2_south:.3f}')

MAE on the south side: 0.567
MSE on the south side: 0.588
R2 on the south side: 0.705


**Verdict:** The errors on both sides are both fairly low. However, the linear relationship between `X` and `y` (represented by `R2`) is slightly more accurate on the north side than the south side.

### Hyperparameter Tuning

In [21]:
mlr_pipeline = Pipeline([('preprocess', preprocessor),
                         ('regr', LinearRegression())])
param_grid = {'preprocess__pca__n_components': [0.90, 0.95, 0.99, None],
              'regr__fit_intercept': [True, False]}
gs = GridSearchCV(mlr_pipeline,
                  param_grid,
                  scoring='neg_mean_squared_error',
                  cv=5,
                  n_jobs=-1)

In [22]:
# 1. The north side
gs.fit(X_train_north_processed, y_train_north)
best_score_north = gs.best_score_
print(best_score_north)
best_params_north = gs.best_params_
print(best_params_north)

-3.5294087142064043
{'preprocess__pca__n_components': 0.99, 'regr__fit_intercept': True}


In [23]:
# 2. The north side
gs.fit(X_train_south_processed, y_train_south)
best_score_south = gs.best_score_
print(best_score_south)
best_params_south = gs.best_params_
print(best_params_south)

-1.9163593382459503
{'preprocess__pca__n_components': 0.95, 'regr__fit_intercept': True}


## Build Machine Learning Pipeline

In [24]:
# 1. Update search spaces from the narrative
search_space_north = [{'regr': [LinearRegression()], 'regr__fit_intercept': [True, False]},
                      {'regr': [Ridge(), Lasso()], 'preprocess__pca__n_components': [0.90, 0.95, 0.99, None]}]
search_space_south = [{'regr': [LinearRegression()], 'regr__fit_intercept': [True, False]},
                      {'regr': [Ridge(), Lasso()], 'preprocess__pca__n_components': [0.90, 0.95, 0.99, None]}]

In [25]:
# 2. Initialise a grid search on search spaces
gs_north = GridSearchCV(mlr_pipeline,
                        param_grid=search_space_north,
                        scoring='neg_mean_squared_error',
                        cv=5)
gs_south = GridSearchCV(mlr_pipeline,
                        param_grid=search_space_south,
                        scoring='neg_mean_squared_error',
                        cv=5)

In [26]:
# 3. Find the best pipeline, regression model, and its hyperparameters on the north and south sides
## i. Fit to training data
gs_north.fit(X_train_north, y_train_north)
gs_south.fit(X_train_south, y_train_south)
## ii. Find the best pipelines
best_pipeline_north = gs_north.best_estimator_
best_pipeline_south = gs_south.best_estimator_
## iii. Find the best regression models
best_regression_model_north = best_pipeline_north.named_steps['regr']
print('The best regression model on the north side is:', best_regression_model_north)
best_regression_model_south = best_pipeline_south.named_steps['regr']
print('The best regression model on the south side is:', best_regression_model_south)
## iv. Find the hyperparameters of the best regression model
best_model_hyperparameters_north = best_regression_model_north.get_params()
print('The hyperparameters on the north side are:', best_model_hyperparameters_north)
best_model_hyperparameters_south = best_regression_model_south.get_params()
print('The hyperparameters on the north side are:', best_model_hyperparameters_south)

C:\Users\Michael\anaconda3\Lib\site-packages\sklearn\impute\_base.py:647: UserWarning: Skipping features without any observed values: ['S1-TAEM']. At least one non-missing value is needed for imputation with strategy='mean'.
  warnings.warn(
C:\Users\Michael\anaconda3\Lib\site-packages\sklearn\impute\_base.py:647: UserWarning: Skipping features without any observed values: ['S1-TAEM']. At least one non-missing value is needed for imputation with strategy='mean'.
  warnings.warn(
C:\Users\Michael\anaconda3\Lib\site-packages\sklearn\impute\_base.py:647: UserWarning: Skipping features without any observed values: ['S1-TAEM']. At least one non-missing value is needed for imputation with strategy='mean'.
  warnings.warn(
C:\Users\Michael\anaconda3\Lib\site-packages\sklearn\impute\_base.py:647: UserWarning: Skipping features without any observed values: ['S1-TAEM']. At least one non-missing value is needed for imputation with strategy='mean'.
  warnings.warn(
C:\Users\Michael\anaconda3\Lib\s

The best regression model on the north side is: Ridge()
The best regression model on the south side is: Ridge()
The hyperparameters on the north side are: {'alpha': 1.0, 'copy_X': True, 'fit_intercept': True, 'max_iter': None, 'positive': False, 'random_state': None, 'solver': 'auto', 'tol': 0.0001}
The hyperparameters on the north side are: {'alpha': 1.0, 'copy_X': True, 'fit_intercept': True, 'max_iter': None, 'positive': False, 'random_state': None, 'solver': 'auto', 'tol': 0.0001}


In [27]:
# 4. Gauge the accuracy of the best pipelines
# i. The north side
y_pred_north = best_pipeline_north.predict(X_test_north)
MAE_north = mean_absolute_error(y_test_north, y_pred_north)
MSE_north = mean_squared_error(y_test_north, y_pred_north)
r2_north = best_pipeline_north.score(X_test_north, y_test_north)
print(f'MAE on the north side: {MAE_north:.3f}')
print(f'MSE on the north side: {MSE_north:.3f}')
print(f'R2 on the north side: {r2_north:.3f}')
# ii. The south side
y_pred_south = best_pipeline_south.predict(X_test_south)
MAE_south = mean_absolute_error(y_test_south, y_pred_south)
MSE_south = mean_squared_error(y_test_south, y_pred_south)
r2_south = best_pipeline_south.score(X_test_south, y_test_south)
print(f'MAE on the south side: {MAE_south:.3f}')
print(f'MSE on the south side: {MSE_south:.3f}')
print(f'R2 on the south side: {r2_south:.3f}')

MAE on the north side: 0.098
MSE on the north side: 0.021
R2 on the north side: 0.989
MAE on the south side: 0.111
MSE on the south side: 0.026
R2 on the south side: 0.973


C:\Users\Michael\anaconda3\Lib\site-packages\sklearn\impute\_base.py:647: UserWarning: Skipping features without any observed values: ['S1-TAEM']. At least one non-missing value is needed for imputation with strategy='mean'.
  warnings.warn(
C:\Users\Michael\anaconda3\Lib\site-packages\sklearn\impute\_base.py:647: UserWarning: Skipping features without any observed values: ['S1-TAEM']. At least one non-missing value is needed for imputation with strategy='mean'.
  warnings.warn(


**Verdict:** Yet again, there is a stronger relationship between moisture content and temperature/humidity on the right side than on the south side. Errors are also lower than the baselines, confirming that tuning improved performance.